# Notebook 14 · CPU inference time in a single session (added after the registered analysis)

**Question:** in the article, the times of RetinaNet and of the YOLO arms were measured in two Colab sessions,
with 6 and 8 PyTorch threads. Here the three arms are measured **in the same session, with the same number of
threads**.

**What it does:** uses the same models (fold 1), the same 100 CT-ICH slices and the same timing function as
notebooks 07 and 09 (`ichdet.final.cpu_latency`: one image at a time, 5 warm-up images). The three arms are
timed in 3 rounds, in rotating order, so that a slower moment of the machine does not favour any of them.
Nothing is trained, and the registered results are not touched (a separate file is written).

**Before running:** the `src/ichdet` folder, version 0.6.0, on the Drive. **Use a CPU runtime** (*Runtime →
Change runtime type → CPU*): the point is to measure without a GPU. **Time:** about 15 to 30 minutes.

In [ ]:
# 1. Drive, libraries and environment
import json, os, platform, subprocess, sys, time, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/ich-benchmark')
    DISCO = Path('/content')                         # Colab disk: reading many images from the Drive is slow
    from importlib.metadata import version
    numpy_antes = version('numpy')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'monai==1.5.2', 'ultralytics>=8.3,<8.4'],
                   check=True)
    if version('numpy') != numpy_antes:
        raise SystemExit(f'O pip trocou o numpy ({numpy_antes} -> {version("numpy")}). '
                         'Reinicie a sessão (Ambiente de execução → Reiniciar sessão) e rode de novo.')
except ImportError:                                  # local test: RAIZ mimics the Drive folder
    RAIZ = Path(os.environ['EXPERIMENTO_LOCAL'])
    DISCO = RAIZ / 'disco_local'

sys.path.insert(0, str(RAIZ / 'codigo'))
PREPARO = RAIZ / 'dados' / 'tc' / 'preparo'
PRIMEIRA = RAIZ / 'dados' / 'tc' / 'experimento' / 'completo'
CORRECAO = RAIZ / 'dados' / 'tc' / 'experimento' / 'correcao_retinanet'
SAIDA = CORRECAO / 'exploratorio'
SAIDA.mkdir(exist_ok=True)
CTICH = DISCO / 'ctich_512'

import numpy as np, torch, monai, ultralytics
import ichdet
from ichdet import final
assert ichdet.__version__ >= '0.6.0', f'ichdet {ichdet.__version__}: copie de novo a pasta src/ichdet do repositório'
assert not torch.cuda.is_available() or 'EXPERIMENTO_LOCAL' in os.environ, (
    'Este notebook mede o tempo em CPU: troque para CPU (Ambiente de execução → Alterar o tipo).')

THREADS = os.cpu_count()                             # the same number for the three arms
torch.set_num_threads(THREADS)
AMBIENTE = {'ichdet_version': ichdet.__version__, 'python': sys.version.split()[0], 'torch': torch.__version__,
            'monai': monai.__version__, 'ultralytics': ultralytics.__version__,
            'cpu': platform.processor() or platform.machine(), 'cpu_count': os.cpu_count(),
            'torch_threads': torch.get_num_threads(), 'started': time.strftime('%Y-%m-%d %H:%M:%S')}
print(json.dumps(AMBIENTE, indent=2))

In [ ]:
# 2. CT-ICH images and fold-1 weights of each arm
if not (CTICH / 'index.csv').exists():
    with zipfile.ZipFile(PREPARO / 'ctich_512.zip') as z:
        z.extractall(CTICH)
teste = json.loads((CTICH / 'monai' / 'external.json').read_text())['validation']

A, B, C = 'retinanet', 'yolo_no_mosaic', 'yolo_mosaic'
PESOS = {A: final.weights_path(CORRECAO, 1, A), B: final.weights_path(PRIMEIRA, 1, B), C: final.weights_path(PRIMEIRA, 1, C)}
for b, p in PESOS.items():
    assert p.exists(), f'{b}: pesos não encontrados em {p}'
print(f'{len(teste)} cortes do CT-ICH; pesos encontrados para os três braços')

In [ ]:
# 3. Three rounds, with the order of the arms rotating each round
ORDENS = [(A, B, C), (B, C, A), (C, A, B)]
voltas = []
for i, ordem in enumerate(ORDENS, 1):
    volta = {}
    for b in ordem:
        torch.set_num_threads(THREADS)
        volta[b] = final.cpu_latency(b, PESOS[b], teste, CTICH)
        print(f"volta {i}: {b:15s} mediana {volta[b]['median_seconds_per_slice'] * 1000:.0f} ms "
              f"(percentil 90 {volta[b]['p90_seconds_per_slice'] * 1000:.0f} ms; {volta[b]['torch_threads']} threads)")
    voltas.append(volta)

In [ ]:
# 4. Summary and output file
resumo = {}
for b in (A, B, C):
    medianas = [v[b]['median_seconds_per_slice'] for v in voltas]
    resumo[b] = {'median_seconds_per_slice': float(np.median(medianas)),
                 'rounds_median_seconds_per_slice': medianas,
                 'p90_seconds_per_slice': float(np.median([v[b]['p90_seconds_per_slice'] for v in voltas])),
                 'torch_threads': voltas[0][b]['torch_threads']}
for b in (A, B, C):
    print(f"{b:15s} mediana das voltas {resumo[b]['median_seconds_per_slice'] * 1000:.0f} ms por corte")
print(f"razão RetinaNet / YOLO sem mosaico: {resumo[A]['median_seconds_per_slice'] / resumo[B]['median_seconds_per_slice']:.2f}")

resultado = {'note': ('CPU inference time of the three arms in one session with the same number of PyTorch threads '
                      '(co-author comment of 20 Nov 2026); fold-1 models, 100 CT-ICH slices, one at a time.'),
             'computed': time.strftime('%Y-%m-%d %H:%M:%S'), 'environment': AMBIENTE,
             'round_orders': [list(o) for o in ORDENS], 'rounds': voltas, 'summary': resumo}
(SAIDA / 'cpu_latency_same_session.json').write_text(json.dumps(resultado, indent=1, default=float))
print('gravado em', SAIDA / 'cpu_latency_same_session.json')

## After running

1. Save the executed notebook (*File → Download → .ipynb*).
2. Download `cpu_latency_same_session.json` from `dados/tc/experimento/correcao_retinanet/exploratorio/` on the
   Drive; it is the file `data/correcao_retinanet/cpu_latency_same_session.json` of this repository.